In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import time

BASE_URL = "https://www.gutenberg.org"

HEADERS = {
    "User-Agent": (
        "FURB-PLN-DataExtraction"
        "(academic project; contact: jectrevisol@furb.br, mrharbs@furb.br, moplgalvao@furb.br)"
    )
}

In [2]:
def get_soup(url):
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    return BeautifulSoup(response.text, "html.parser")

def load_existing_data():
    try:
        df = pd.read_csv(
            "gutenberg_books.csv",
            dtype={"book_id": str}
        )

        return df

    except FileNotFoundError:
        return pd.DataFrame()


def get_books_from_category(category_url):

    soup = get_soup(category_url)

    books = []

    for link in soup.find_all("a", href=True):

        href = link["href"]

        match = re.fullmatch(r"/ebooks/(\d+)", href)

        if not match:
            continue

        book_id = match.group(1)

        title = link.get_text(" ", strip=True)

        books.append({
            "book_id": book_id,
            "title": title,
            "ebook_url": urljoin(BASE_URL, href)
        })

    next_url = None

    for link in soup.find_all("a", href=True):

        text = link.get_text(" ", strip=True)

        if text == "Next":
            next_url = urljoin(BASE_URL, link["href"])
            break

    return books, next_url

def get_metadata_table(soup):
    metadata = {}

    for row in soup.select("table tr"):
        cells = row.find_all(["th", "td"])

        if len(cells) < 2:
            continue

        key = cells[0].get_text(" ", strip=True)
        value = cells[1].get_text(" ", strip=True)

        if key == "Subject":
            metadata.setdefault("Subject", []).append(value)
        else:
            metadata[key] = value

    return metadata

def get_summary(soup):
    page_body = soup.find("div", class_="page-body")

    if not page_body:
        return None

    text = page_body.get_text(" ", strip=True)

    text = re.sub(r"^QR code\s*", "", text)

    summary = text.split("Read more", 1)[0]

    summary = re.sub(
        r"\(This is an automatically generated summary\.\)",
        "",
        summary
    )

    return summary.strip()

def normalize_text(text):

    if not text:
        return ""

    text = text.lower()

    import unicodedata

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        char
        for char in text
        if not unicodedata.combining(char)
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def create_work_key(title, author):

    normalized_title = normalize_text(
        title
    )

    normalized_author = normalize_text(
        author
    )

    return (
        normalized_title
        + "|"
        + normalized_author
    )


def get_book_metadata(book):

    print(
        f"Coletando: {book['title']}"
    )

    soup = get_soup(
        book["ebook_url"]
    )

    metadata = get_metadata_table(
        soup
    )

    language = metadata.get(
        "Language"
    )

    if language != "English":

        print(
            f"Ignorado: idioma = {language}"
        )

        return None

    title = metadata.get(
        "Title",
        book["title"]
    )

    author = metadata.get(
        "Author"
    )

    data = {

        "book_id": book["book_id"],

        "title": title,

        "author": author,

        "language": language,

        "loc_class": metadata.get(
            "LoC Class"
        ),

        "subjects": metadata.get(
            "Subject",
            []
        ),

        "release_date": metadata.get(
            "Release Date"
        ),

        "ebook_url": book["ebook_url"],

        "summary": get_summary(
            soup
        )
    }

    data["subjects"] = " | ".join(
        data["subjects"]
    )

    data["work_key"] = create_work_key(
        title,
        author
    )

    return data

In [6]:
def main():

    category_url = (
        "https://www.gutenberg.org/ebooks/bookshelf/639"
    )

    df_existing = load_existing_data()

    existing_ids = set()

    existing_work_keys = set()

    if not df_existing.empty:

        if "book_id" in df_existing.columns:

            existing_ids = set(
                df_existing["book_id"]
                .astype(str)
            )

        if "work_key" in df_existing.columns:

            existing_work_keys = set(
                df_existing["work_key"]
                .dropna()
                .astype(str)
            )

        elif (
            "title" in df_existing.columns
            and "author" in df_existing.columns
        ):

            for _, row in df_existing.iterrows():

                work_key = create_work_key(
                    row["title"],
                    row["author"]
                )

                existing_work_keys.add(
                    work_key
                )

    page_number = 1

    visited_pages = set()

    while category_url:

        # aqui ta limitando a execução as 3 primeiras paginas, se executar denovo do msm jeito que ta, ele nao vai pegar nenhum livro,
        # precisa ir aumentando conforme vai executando, precisa executar em partes para nao estourar o limite de requests do gutenberg
        # próx execução, podemos ir aumentando de 3 em 3, só q se executa ate as 6 paginas, dps tem q esperar um tempo pra executar dnv e tal
        if page_number > 4:
            break

        if category_url in visited_pages:

            print(
                "Página já visitada. "
                "Encerrando paginação."
            )

            break

        visited_pages.add(
            category_url
        )

        print(
            f"\n--- Página {page_number} ---"
        )

        print(
            f"URL: {category_url}"
        )

        try:

            books, next_url = (
                get_books_from_category(
                    category_url
                )
            )

        except requests.RequestException as error:

            print(
                f"Erro ao acessar página: {error}"
            )

            break

        print(
            f"Livros encontrados: {len(books)}"
        )

        for book in books:

            book_id = str(
                book["book_id"]
            )

            if book_id in existing_ids:

                print(
                    f"Já coletado: "
                    f"{book['title']} "
                    f"(ID {book_id})"
                )

                continue

            try:

                data = get_book_metadata(
                    book
                )

                if data is None:

                    existing_ids.add(
                        book_id
                    )

                    time.sleep(5)

                    continue

                work_key = data[
                    "work_key"
                ]

                if work_key in existing_work_keys:

                    print(
                        "Ignorado: obra já "
                        "coletada com outro ID"
                    )

                    existing_ids.add(
                        book_id
                    )

                    time.sleep(5)

                    continue

                new_row = pd.DataFrame(
                    [data]
                )

                df_existing = pd.concat(
                    [
                        df_existing,
                        new_row
                    ],
                    ignore_index=True
                )

                df_existing.to_csv(
                    "gutenberg_books.csv",
                    index=False,
                    encoding="utf-8-sig"
                )

                existing_ids.add(
                    book_id
                )

                existing_work_keys.add(
                    work_key
                )

                print(
                    f"Salvo: {data['title']}"
                )

                time.sleep(5)

            except requests.RequestException as error:

                print(
                    f"Erro ao coletar "
                    f"{book['title']}: {error}"
                )

        category_url = next_url

        page_number += 1

        if category_url:

            time.sleep(5)

    print(
        "\nColeta finalizada."
    )

    print(
        f"Total de livros no CSV: "
        f"{len(df_existing)}"
    )


if __name__ == "__main__":
    main()


--- Página 1 ---
URL: https://www.gutenberg.org/ebooks/bookshelf/639
Livros encontrados: 25
Já coletado: Pride and Prejudice Jane Austen 175901 downloads (ID 1342)
Coletando: Romeo and Juliet William Shakespeare 89335 downloads
Salvo: Romeo and Juliet
Já coletado: A Room with a View E. M. Forster 78526 downloads (ID 2641)
Já coletado: The Secret of Chimneys Agatha Christie 73123 downloads (ID 65238)
Já coletado: The Mysteries of Udolpho Ann Ward Radcliffe 71056 downloads (ID 3268)
Já coletado: The Blue Castle: a novel L. M. Montgomery 67153 downloads (ID 67979)
Já coletado: Jane Eyre: An Autobiography Charlotte Brontë 66416 downloads (ID 1260)
Já coletado: Carmen Prosper Mérimée 64795 downloads (ID 2465)
Já coletado: The String of Pearls; Or, The Barber of Fleet Street. A Domestic Romance. Thomas Peckett Prest and James Malcolm Rymer 56189 downloads (ID 59828)
Já coletado: The Green Mummy Fergus Hume 55959 downloads (ID 2868)
Já coletado: Le Fantôme de l'Opéra (French) Gaston Leroux 5

In [7]:
import pandas as pd

df = pd.read_csv("gutenberg_books.csv")

df

,book_id,title,author,language,loc_class,subjects,release_date,ebook_url,summary,work_key,trecho
0,1342,Pride and Prejudice,"Austen, Jane, 1775-1817",English,PR: Language and Literatures: English literature,England -- Fiction | Young women -- Fiction | ...,"Jun 1, 1998",https://www.gutenberg.org/ebooks/1342,"""Pride and Prejudice"" by Jane Austen is a nove...",NaN,"Mrs. Gardiner had seen Pemberley, and known th..."
1,2701,"Moby Dick; Or, The Whale","Melville, Herman, 1819-1891",English,PS: Language and Literatures: American and Can...,Whaling -- Fiction | Sea stories | Psychologic...,"Jul 1, 2001",https://www.gutenberg.org/ebooks/2701,"""Moby Dick; Or, The Whale"" by Herman Melville ...",NaN,The acute policy dictating these movements was...
2,2554,Crime and Punishment,"Dostoyevsky, Fyodor, 1821-1881",English,PG: Language and Literatures: Slavic (includin...,Detective and mystery stories | Psychological ...,"Mar 28, 2006",https://www.gutenberg.org/ebooks/2554,"""Crime and Punishment"" by Fyodor Dostoevsky is...",NaN,These exclamations and remarks checked Raskoln...
3,84,"Frankenstein; or, the modern prometheus","Shelley, Mary Wollstonecraft, 1797-1851",English,PR: Language and Literatures: English literature,Science fiction | Horror tales | Gothic fictio...,"Oct 1, 1993",https://www.gutenberg.org/ebooks/84,"""Frankenstein; Or, The Modern Prometheus"" by M...",NaN,“Such were the events that preyed on the heart...
4,11,Alice's Adventures in Wonderland,"Carroll, Lewis, 1832-1898",English,PZ: Language and Literatures: Juvenile belles ...,Fantasy fiction | Children's stories | Imagina...,"Jun 27, 2008",https://www.gutenberg.org/ebooks/11,"""Alice's Adventures in Wonderland"" by Lewis Ca...",NaN,“What _can_ all that green stuff be?” said Ali...
...,...,...,...,...,...,...,...,...,...,...,...
307,76668,Trial by water,"Wright, Sewell Peaslee, 1897-1970",English,PS: Language and Literatures: American and Can...,Short stories | Triangles (Interpersonal relat...,"Aug 10, 2025",https://www.gutenberg.org/ebooks/76668,"""Trial by water by Sewell Peaslee Wright"" is a...",trial by water|wright sewell peaslee 1897 1970,NaN
308,9473,"The Knights of the Cross, or, Krzyzacy: Histor...","Sienkiewicz, Henryk, 1846-1916",English,PG: Language and Literatures: Slavic (includin...,Teutonic Knights -- History -- Fiction | Polan...,"Dec 1, 2005",https://www.gutenberg.org/ebooks/9473,"""The Knights of the Cross, or, Krzyzacy: Histo...",the knights of the cross or krzyzacy historica...,NaN
309,60316,The Bakhtyār Nāma: A Persian Romance,NaN,English,PK: Language and Literatures: Indo-Iranian lit...,Persian poetry -- Translations into English,"Sep 17, 2019",https://www.gutenberg.org/ebooks/60316,"""The Bakhtyār Nāma: A Persian Romance"" by Will...",the bakhtyar nama a persian romance|,NaN
310,19146,"The Entailed Hat; Or, Patty Cannon's Times","Townsend, George Alfred, 1841-1914",English,PS: Language and Literatures: American and Can...,Kidnapping -- Fiction | Eastern Shore (Md. and...,"Aug 30, 2006",https://www.gutenberg.org/ebooks/19146,"""The Entailed Hat; Or, Patty Cannon's Times"" b...",the entailed hat or patty cannon s times|towns...,NaN
